In [ ]:
ALGORITHM    = "substitution"
GENERATE_MIP = False

import re
import subprocess
import sys
import json
import tempfile
from pathlib import Path

PROJECT_ROOT = Path("../..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import MODELS, PAPILO_PATH
from src.presolvers.papilo import _ALL_PAPILO_PRESOLVERS
from tools.parse_papilo_log import parse, to_metadata

_suffix = "_mip" if GENERATE_MIP else ""

STATIC_OUT = Path(f"/data/energy-system-preprocessing/presolve/static{_suffix}") / ALGORITHM
STATIC_OUT.mkdir(parents=True, exist_ok=True)


def _is_target(name: str) -> bool:
    if not re.match(r"^r\d+_", name):
        return False
    return name.endswith("_mip_obj") if GENERATE_MIP else (not name.endswith("_mip_obj") and name.endswith("_obj"))


assert ALGORITHM in _ALL_PAPILO_PRESOLVERS, (
    f"Unknown algorithm '{ALGORITHM}'. "
    f"Valid options: {sorted(_ALL_PAPILO_PRESOLVERS)}"
)

print(f"Algorithm : {ALGORITHM}")
print(f"MIP models: {GENERATE_MIP}")
print(f"Output dir: {STATIC_OUT}")
print(f"Disabled  : {sorted(_ALL_PAPILO_PRESOLVERS - {ALGORITHM})}")

Algorithm : substitution
MIP models: False
Output dir: /data/energy-system-preprocessing/presolve/static/substitution
Disabled  : ['cliquemerging', 'coefftightening', 'colsingleton', 'domcol', 'doubletoneq', 'dualfix', 'dualinfer', 'fixcontinuous', 'implint', 'parallelcols', 'parallelrows', 'probing', 'propagation', 'simpleprobing', 'simplifyineq', 'sparsify', 'stuffing']


In [ ]:
def run_static_presolve(model_dir: Path) -> bool:
    model_name = model_dir.name
    mps_in = model_dir / "original.mps"
    if not mps_in.exists():
        print(f"  SKIP: no original.mps in {model_dir}")
        return False

    out_dir = STATIC_OUT / model_name
    out_dir.mkdir(parents=True, exist_ok=True)

    reduced   = out_dir / "reduced.mps"
    postsolve = out_dir / "reduced.postsolve"
    log_file  = out_dir / "output.txt"

    if reduced.exists():
        print(f"  skipped (already exists)", flush=True)
        return True

    disabled = _ALL_PAPILO_PRESOLVERS - {ALGORITHM}
    settings_lines = [f"{p}.enabled = 0" for p in sorted(disabled)]

    with tempfile.NamedTemporaryFile(mode="w", suffix=".set", delete=False) as tf:
        tf.write("\n".join(settings_lines) + "\n")
        settings_path = Path(tf.name)

    cmd = [
        str(PAPILO_PATH), "presolve",
        "-f", str(mps_in),
        "-r", str(reduced),
        "-v", str(postsolve),
        "-p", str(settings_path),
        "--message.verbosity=4",
    ]

    try:
        with log_file.open("w") as log:
            process = subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT)
    finally:
        settings_path.unlink(missing_ok=True)

    if process.returncode != 0 or not reduced.exists():
        print(f"  FAILED (rc={process.returncode})")
        return False

    rounds, model_path = parse(log_file.read_text().splitlines(keepends=True))
    meta = to_metadata(rounds, model_path)

    src_meta_path = model_dir / "metadata.json"
    existing: dict = json.loads(src_meta_path.read_text()) if src_meta_path.exists() else {}
    existing.update(meta)
    (out_dir / "metadata.json").write_text(json.dumps(existing, indent=2))

    return True

In [3]:
models = sorted(p for p in MODELS.iterdir() if _is_target(p.name))
print(f"{len(models)} models found in {MODELS}")

failed = []

for i, model_dir in enumerate(models, 1):
    print(f"[{i}/{len(models)}] {model_dir.name}", flush=True)
    ok = run_static_presolve(model_dir)
    if not ok:
        failed.append(model_dir.name)
    print(flush=True)

if failed:
    print(f"\n{len(failed)} model(s) failed: {failed}")
else:
    print(f"\nAll {len(models)} models presolved successfully.")

485 models found in /data/energy-system-preprocessing/models
[1/485] r10_res168_f0.0000_t0.0192_obj
  skipped (already exists)

[2/485] r10_res168_f0.0000_t0.0833_obj
  skipped (already exists)

[3/485] r10_res168_f0.0000_t1.0000_obj
  skipped (already exists)

[4/485] r10_res168_f0.2500_t0.5000_obj
  skipped (already exists)

[5/485] r10_res168_f0.5000_t1.0000_obj
  skipped (already exists)

[6/485] r10_res1_f0.0000_t0.0192_cf0.75_obj
  skipped (already exists)

[7/485] r10_res1_f0.0000_t0.0192_obj
  skipped (already exists)

[8/485] r10_res1_f0.0000_t0.0833_cf0.75_obj
  skipped (already exists)

[9/485] r10_res1_f0.0000_t0.0833_obj
  skipped (already exists)

[10/485] r10_res1_f0.0000_t1.0000_cf0.75_obj
  skipped (already exists)

[11/485] r10_res1_f0.0000_t1.0000_obj
  skipped (already exists)

[12/485] r10_res1_f0.2500_t0.5000_cf0.75_obj
  skipped (already exists)

[13/485] r10_res1_f0.2500_t0.5000_obj
  skipped (already exists)

[14/485] r10_res1_f0.5000_t1.0000_cf0.75_obj
  skipp